In [1]:
import pandas as pd, numpy as np
from pathlib import Path
from datetime import datetime
from unidecode import unidecode
import sys, os

src = Path.cwd().parent/"src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))
pd.set_option('display.max_columns', None)

In [2]:
from routes.Paths import(
    Ruta_Bonos,
    Ruta_global,
    Ruta_local
)
from utils.functions import load_files_from_folder

from utils.Data_ModeloComercial.b_processing_crm_ventas import process_crm_ventas

df_crm = process_crm_ventas(save_file = False)

In [32]:
Required_columns = [
        "supplier",
        "order_id",
        "order_date",
        "numero documento"
]

df_bonos = load_files_from_folder(
    Ruta_Bonos, 
    set(Required_columns),
    Required_columns)

columns_bonos = [
    "supplier",
    "order_id",
    "order_date",
    "numero documento",
    "puntos",
    "source_file",
    "file_date"
]
columnas_crm_requeridas = [
"NÚMERO_DOCUMENTO_DEL_CLIENTE",
"RADICADO",
"IdTipificacion",
"NIVEL_1",
"NIVEL_2",
"NIVEL_3",
"TIPO_DE_LLAMADA",
"NOMBRE_CLIENTE",
"CONTACTO_DE_CLIENTE",
"NOMBRE_ALIADO",
"Id_opl",
"NOMBRE_ASESOR",
"Correo_contratista",
"FECHA_DE_INICIO",
"START_DATE",
"FECHA_FIN",
"END_DATE",
"START_MONTH_DATE",
"OFRECIMIENTO_EFECTIVO",
"Aplica_Ofrecimiento",
"VALOR_PUNTOS",
"CANAL_DE_ATENCIÓN",
"Colas",
"SKILL",
"IdMasterSkill",
"ESTADO_GESTIÓN",
"NÚMERO_PEDIDO"
]

In [33]:
df_bonos = df_bonos[columns_bonos]
df_bonos["order_date"] = pd.to_datetime(df_bonos["order_date"],
                                        format="mixed",
                                        utc=True, 
                                        errors="coerce").dt.tz_localize(None)

df_bonos["date"] = df_bonos["order_date"].dt.date
df_bonos = df_bonos.sort_values(by=["numero documento", "order_date"], ascending=[True, False]).reset_index(drop=True)
df_bonos["numero documento"] = df_bonos["numero documento"].astype(str) 

In [34]:
#Ajuste df CRM Ventas
df_crm["FECHA_DE_INICIO"] = pd.to_datetime(
    df_crm["FECHA_DE_INICIO"],
    errors="coerce"
)

df_crm["FECHA_FIN"] = pd.to_datetime(
    df_crm["FECHA_FIN"],
    errors="coerce"
)

df_crm["Prioridad"] = (
    df_crm["OFRECIMIENTO_EFECTIVO"]
        .map({"Si": 1, "No": 2})
)

df_crm = df_crm.sort_values(
    by=[
        "NÚMERO_DOCUMENTO_DEL_CLIENTE",
        "START_DATE",
        "Prioridad"
    ]
)

In [ ]:
df_bonos = df_bonos.merge(
    df_crm[columnas_crm_requeridas],
    how="left",
    left_on=[
        "numero documento",
        "date"
    ],
    right_on=[
        "NÚMERO_DOCUMENTO_DEL_CLIENTE",
        "START_DATE"
    ]
).drop(columns=["NÚMERO_DOCUMENTO_DEL_CLIENTE","START_DATE"])

df_bonos["HORAS_DESDE_INICIO"] = ((df_bonos["order_date"] - df_bonos["FECHA_DE_INICIO"]).dt.total_seconds()/3600)
df_bonos["HORAS_DESDE_FIN"] = ((df_bonos["order_date"] - df_bonos["FECHA_FIN"]).dt.total_seconds()/3600)

df_bonos["VALIDACION_VENTA"] = np.where(
    (df_bonos["OFRECIMIENTO_EFECTIVO"].eq("Si")) & 
    (df_bonos["HORAS_DESDE_INICIO"].between(0, 24) | df_bonos["HORAS_DESDE_FIN"].between(0, 24)),
    "Venta efectiva",
    "Venta no efectiva"
)

In [37]:
df_bonos.to_csv(os.path.join(Ruta_local, "11.Data_Bonos_Ventas.csv"), index=False)